# H-003 · Kalman β vs static OLS hedge (trad-z)

Both arms use traditional z entry/exit (`k=2 / 0`). Kalman uses store defaults `delta=1e-4`, `obs_var=1e-3`, `burn_in=30` days (session-scaled on 1H). Spread from the **prior** state.

Rolling ADF / half-life on each hedge is a **diagnostic**, not a freeze input. Type `HEDGE_STAR` (`"ols"` or `"kalman"`).


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_c_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
PAIRS_STAR = list(stack["PAIRS_STAR"])
print("stack keys:", sorted(stack))
print("PAIRS_STAR", PAIRS_STAR)


## 1. Load frozen panel / PAIRS_STAR


In [ ]:
require_star("BAR_STAR", stack.get("BAR_STAR"))
BAR = str(stack["BAR_STAR"])
train, full = load_universe_c_panels(BAR, PAIRS_STAR, root=ROOT)
# Keep OLS columns so the bake-off stays honest even if HEDGE_STAR was already typed.
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS rows", len(is_panel), "OOS rows", len(oos_panel))
is_panel.head()


## 2. Attach Kalman hedge columns (OLS panel already loaded)


In [ ]:
lb = lookbacks_for_bar(BAR)
kalman_panel = overlay_kalman_hedge(
    is_panel.copy(),
    burn_in=lb["kalman_burn_in"],
    z_window=lb["z_window"],
    hl_window=lb["hl_window"],
)
diag = pd.DataFrame(
    {
        "ols_median_adf": is_panel.groupby("pair_id")["adf_pvalue"].median(),
        "kf_median_adf": kalman_panel.groupby("pair_id")["adf_pvalue"].median(),
        "ols_median_hl": is_panel.groupby("pair_id")["half_life"].median(),
        "kf_median_hl": kalman_panel.groupby("pair_id")["half_life"].median(),
    }
)
print("ADF/HL diagnostic (not a freeze input)")
diag


## 3. Walk-forward folds


In [ ]:
dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
fold_table(folds)


## 4. Fold-val metrics (validation only)


In [ ]:
cfg_ols = config_from_stack(stack, hedge="ols")
cfg_kf = config_from_stack(stack, hedge="kalman")
df_ols = fold_val_metrics(is_panel, folds, {"ols": cfg_ols}, s1_weekly=s1_weekly)
df_kf = fold_val_metrics(kalman_panel, folds, {"kalman": cfg_kf}, s1_weekly=s1_weekly)
fold_df = pd.concat([df_ols, df_kf], ignore_index=True)
fold_df


## 5. Boxplots (do not assign STAR here)


In [ ]:
plot_fold_boxplots(fold_df, title="H-003 OLS vs Kalman")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
fold_df.groupby("arm")[["ann_sharpe", "max_drawdown", "corr_to_s1"]].median()


## 6. Type `HEDGE_STAR` then save


In [ ]:
HEDGE_STAR = None  # TODO set after review: "ols" or "kalman"
require_star("HEDGE_STAR", HEDGE_STAR)
stack["HEDGE_STAR"] = HEDGE_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH)


## 7. Sealed OOS once


In [ ]:
require_star("HEDGE_STAR", HEDGE_STAR)
oos_use = oos_panel
if HEDGE_STAR == "kalman":
    oos_use = overlay_kalman_hedge(oos_panel.copy(), burn_in=lb["kalman_burn_in"], z_window=lb["z_window"], hl_window=lb["hl_window"])
oos = run_s2_backtest(oos_use, config_from_stack(stack), s1_weekly=s1_weekly)
print(oos.metrics)
write_tearsheet_pdf(tearsheet_path("H-003", HEDGE_STAR), oos.returns, title=f"H-003 {HEDGE_STAR} sealed OOS")


## 8. Notes for next hyp


H-004 cointegration-break flat rule (`off` | `block_05_flat_10` | `flat_05`). Leave `HEDGE_STAR` frozen.
